# Validation Example: Plane Poiseuille Flow

This notebook demonstrates how to validate the `fem` module against the
**plane Poiseuille flow** benchmark — a pressure-driven viscous channel flow
with a known exact analytical solution.

It also shows how to write and run a `unittest` suite directly inside a
Jupyter notebook, which is the recommended approach for interactive validation
examples in this documentation.

---

## Problem description

A pressure difference $\Delta P = p_{\mathrm{in}} - p_{\mathrm{out}}$ drives
viscous flow between two stationary parallel plates separated by height $b$.

| Parameter | Symbol | Value |
|-----------|--------|-------|
| Channel length | $a$ | 6 |
| Channel height | $b$ | 2 |
| Density | $\rho$ | 1 |
| Dynamic viscosity | $\mu$ | 1 |
| Inlet pressure | $p_{\mathrm{in}}$ | 10 |
| Outlet pressure | $p_{\mathrm{out}}$ | 4 |
| Element type | — | Q9 (order = 2) |
| Mesh | — | 4 × 4 elements |

**Boundary conditions**

- Top and bottom walls: no-slip Dirichlet, $\mathbf{u} = \mathbf{0}$
- Left boundary ($x = 0$): pressure inlet $p = p_{\mathrm{in}}$ (Neumann)
- Right boundary ($x = a$): pressure outlet $p = p_{\mathrm{out}}$ (Neumann)

### Analytical solution

Defining the streamwise pressure gradient
$$
\frac{\mathrm{d}p}{\mathrm{d}x} = \frac{p_{\mathrm{out}} - p_{\mathrm{in}}}{a} < 0,
$$
the exact solution is
$$
v_x(x,y) = -\frac{1}{2\mu}\frac{\mathrm{d}p}{\mathrm{d}x}\, y\,(b - y),
\qquad
v_y = 0,
\qquad
p(x,y) = p_{\mathrm{in}} + \frac{\mathrm{d}p}{\mathrm{d}x}\, x.
$$

The peak (centreline) velocity is
$$
v_{x,\max} = -\frac{\mathrm{d}p}{\mathrm{d}x}\frac{b^2}{8\mu},
$$
and the volumetric flow rate per unit width is
$$
Q = \int_0^b v_x \, \mathrm{d}y = \frac{2}{3}\, v_{x,\max}\, b.
$$

## 1 - Imports

In [ ]:
import unittest
import numpy as np
import matplotlib.pyplot as plt

from fem import NavierStokesSolver, BoundaryCondition, BCType, BCVar

plt.rcParams.update({'font.size': 13, 'font.family': 'serif',
                     'mathtext.fontset': 'cm'})

## 2 - Problem setup and solver

We build the mesh, assign the physical parameters, and apply the boundary
conditions. The solver is stored in a module-level variable so the FEM
solve only runs once — the same pattern used by `unittest.TestCase.setUpClass`.

In [ ]:
# ── Domain ────────────────────────────────────────────────────────────────────
a, b   = 6, 2          # channel length, height
nx = ny = 4            # elements per direction
order  = 2             # Q9 elements

# ── Physics ───────────────────────────────────────────────────────────────────
rho, mu = 1.0, 1.0

# ── Pressure values ───────────────────────────────────────────────────────────
p_in  = 10.0
p_out = 4.0
dPdx  = (p_out - p_in) / a    # pressure gradient (< 0 → flow in +x)

print(f"Pressure gradient  dP/dx = {dPdx:.4f}")
print(f"Peak velocity  vx_max    = {-dPdx * b**2 / (8*mu):.4f}")

In [ ]:
# ── Boundary conditions ───────────────────────────────────────────────────────
top = BoundaryCondition(
    name="no-slip-top",
    boundary_key="top",
    bc_type=BCType.DIRICHLET,
    variable=BCVar.VELOCITY,
    value=(0.0, 0.0),
    apply_strong=True,
    metadata={"note": "no-slip top wall"},
)
bottom = BoundaryCondition(
    name="no-slip-bottom",
    boundary_key="bottom",
    bc_type=BCType.DIRICHLET,
    variable=BCVar.VELOCITY,
    value=(0.0, 0.0),
    apply_strong=True,
    metadata={"note": "no-slip bottom wall"},
)
inlet = BoundaryCondition(
    name="pressure-inlet",
    boundary_key="left",
    bc_type=BCType.NEUMANN,
    variable=BCVar.PRESSURE,
    value=p_in,
    apply_strong=False,
    metadata={"p": p_in},
)
outlet = BoundaryCondition(
    name="pressure-outlet",
    boundary_key="right",
    bc_type=BCType.NEUMANN,
    variable=BCVar.PRESSURE,
    value=p_out,
    apply_strong=False,
    metadata={"p": p_out},
)

# ── Solve ─────────────────────────────────────────────────────────────────────
sol = NavierStokesSolver.uniform_rectangular_domain_rect(
    nx, ny, a, b, order=order
)
sol.setup_physics(rho, mu)
sol.setup_boundary_conditions([bottom, top, inlet, outlet])
sol.solve_steadystate(u0=1, p0=p_in)

sol_vx, sol_vy, sol_p = sol.get_solution()
print("Solver finished.  Solution arrays:",
      sol_vx.shape, sol_vy.shape, sol_p.shape)

## 3 - Analytical solution helpers

In [ ]:
def vx_analytical(x, y):
    """Parabolic Poiseuille profile."""
    return (-1.0 / (2.0 * mu)) * dPdx * y * (b - y)

def vy_analytical(x, y):
    """Zero vertical velocity."""
    return np.zeros_like(np.asarray(y, dtype=float))

def p_analytical(x, y):
    """Linear pressure from inlet to outlet."""
    return p_in + dPdx * np.asarray(x, dtype=float)

vx_max = -dPdx * b**2 / (8.0 * mu)
print(f"vx_max = {vx_max:.6f}")

## 4 - Quick sanity check

`np.allclose` gives an immediate pass/fail before running the formal test suite.

In [ ]:
nodes_v = sol.p2_nodes   # velocity nodes (P2 / Q9)
nodes_p = sol.p1_nodes   # pressure nodes (P1 / Q4)

vx_ok = np.allclose(sol_vx, vx_analytical(nodes_v[:, 0], nodes_v[:, 1]))
vy_ok = np.allclose(sol_vy, vy_analytical(nodes_v[:, 0], nodes_v[:, 1]))
p_ok  = np.allclose(sol_p,  p_analytical( nodes_p[:, 0], nodes_p[:, 1]))

print(f"vx matches analytical parabola : {vx_ok}")
print(f"vy == 0 everywhere             : {vy_ok}")
print(f"p  matches linear profile      : {p_ok}")

## 5 - Visualisation

The plots below give a visual confirmation that matches the quantitative tests.

In [ ]:
# ── collect nodes at x = 0 and x = a ─────────────────────────────────────────
uni_x   = sol.group_by_x()
x_stations = {k: v for k, v in uni_x.items() if k in [0.0, a]}

markers    = ['o', 's', '^', 'd']
linestyles = ['--', '-', '-.']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── (a) velocity profiles ─────────────────────────────────────────────────────
ax = axes[0]
y_fine = np.linspace(0, b, 200)
ax.plot(vx_analytical(a, y_fine), y_fine, 'r', lw=2, label='Analytical')
for i, (xs, con) in enumerate(x_stations.items()):
    ys = sol.p2_nodes[con, 1]
    ax.plot(sol_vx[con], ys, 'k', marker=markers[i], linestyle=linestyles[i],
            ms=8, markerfacecolor='none', label=f'FEM  $x={xs:.0f}$')
ax.set_xlabel('$v_x(y)$')
ax.set_ylabel('$y$', rotation=0, labelpad=10)
ax.set_title('(a) Velocity profile')
ax.legend(fontsize=11)
ax.grid()

# ── (b) pressure profile (along y at x = a/2) ─────────────────────────────────
ax = axes[1]
# all pressure nodes, plotted vs x
px = sol.p1_nodes[:, 0]
x_fine = np.linspace(0, a, 200)
ax.plot(x_fine, p_analytical(x_fine, 0), 'r', lw=2, label='Analytical')
ax.scatter(px, sol_p, c='k', s=30, zorder=3, label='FEM nodes')
ax.set_xlabel('$x$')
ax.set_ylabel('$p$', rotation=0, labelpad=10)
ax.set_title('(b) Pressure distribution')
ax.legend(fontsize=11)
ax.grid()

# ── (c) centreline velocity vs analytical ─────────────────────────────────────
ax = axes[2]
centre_idx = np.where(np.abs(sol.p2_nodes[:, 1] - b / 2.0) < 1e-6)[0]
xc = sol.p2_nodes[centre_idx, 0]
ax.axhline(vx_max, color='r', lw=2, label=f'$v_{{x,\\max}}={vx_max:.3f}$')
ax.scatter(xc, sol_vx[centre_idx], c='k', s=40, zorder=3, label='FEM centreline')
ax.set_xlabel('$x$')
ax.set_ylabel('$v_x$', rotation=0, labelpad=10)
ax.set_title('(c) Centreline velocity')
ax.legend(fontsize=11)
ax.grid()

fig.suptitle('Plane Poiseuille Flow — FEM vs Analytical', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 8 — Test summary

| Test | What it checks |
|------|----------------|
| `test_vx_matches_analytical` | Full $v_x$ field matches $-\frac{1}{2\mu}\frac{dp}{dx}y(b-y)$ at every node |
| `test_vy_is_zero` | $v_y = 0$ everywhere |
| `test_pressure_linear_in_x` | Full pressure field matches $p_{\mathrm{in}} + \frac{dp}{dx}x$ |
| `test_pressure_inlet_value` | $p = p_{\mathrm{in}}$ at $x = 0$ |
| `test_pressure_outlet_value` | $p = p_{\mathrm{out}}$ at $x = a$ |
| `test_no_slip_top_wall` | Top wall enforces $v_x = v_y = 0$ |
| `test_no_slip_bottom_wall` | Bottom wall enforces $v_x = v_y = 0$ |
| `test_peak_velocity_at_centreline` | $v_x(b/2) = v_{x,\max}$ |
| `test_velocity_profile_parabolic` | $v_x(y)$ fits a degree-2 polynomial with $R^2 \geq 0.9999$ |
| `test_velocity_profile_symmetric` | $v_x(x, y) = v_x(x, b-y)$ (symmetry about centreline) |
| `test_flow_rate_matches_analytical` | $Q = \int_0^b v_x \, dy = \tfrac{2}{3} v_{x,\max} b$ |
| `test_solution_shape` | Array sizes consistent with node counts |

All tests pass when the FEM solver correctly recovers the Poiseuille flow solution.

---
*To run these tests from the command line, use the standalone script `test_poiseuille_flow.py`.*